In [33]:
# CELL 1 — Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [64]:
!pip install -q pymupdf faiss-cpu sentence-transformers transformers accelerate bitsandbytes gradio

In [65]:
import os
import torch
import faiss
import numpy as np
import fitz
import shutil

from pathlib import Path
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [66]:
# ==========================================
# CELL 4 — CREATE PROJECT FOLDERS
# ==========================================

base_path = "/content/drive/MyDrive/STUDYMATE_AI"

folders = [
    "uploads",
    "extracted_text",
    "vector_db",
    "outputs",
    "audio",
    "youtube"
]

for folder in folders:
    os.makedirs(f"{base_path}/{folder}", exist_ok=True)

print("PROJECT FOLDERS CREATED")

PROJECT FOLDERS CREATED


In [67]:
# ==========================================
# CELL 5 — UPLOAD FILE
# ==========================================

uploaded = files.upload()

for file in uploaded.keys():
    shutil.move(file, f"{base_path}/uploads/{file}")

print("FILE UPLOADED SUCCESSFULLY")
print(os.listdir(f"{base_path}/uploads"))

Saving nlp_lecture1_1 (1).pdf to nlp_lecture1_1 (1).pdf
FILE UPLOADED SUCCESSFULLY
['nlp_lecture1_1 (1).pdf']


In [68]:
# ==========================================
# CELL 6 — SELECT LATEST PDF
# ==========================================

upload_folder = f"{base_path}/uploads"

pdf_files = [
    f for f in os.listdir(upload_folder)
    if f.lower().endswith(".pdf")
]

pdf_files = sorted(
    pdf_files,
    key=lambda x: os.path.getmtime(f"{upload_folder}/{x}"),
    reverse=True
)

pdf_path = f"{upload_folder}/{pdf_files[0]}"

print("SELECTED PDF:")
print(pdf_path)

SELECTED PDF:
/content/drive/MyDrive/STUDYMATE_AI/uploads/nlp_lecture1_1 (1).pdf


In [69]:
# ==========================================
# CELL 7 — DEFINE STUDYMATE RAG PIPELINE
# ==========================================

class StudyMateRAG:
    def __init__(
        self,
        embedding_model_name="BAAI/bge-m3",
        llm_model_name="Qwen/Qwen2.5-1.5B-Instruct",
        chunk_size=700,
        chunk_overlap=120,
        top_k=5,
        use_4bit=True
    ):
        self.embedding_model_name = embedding_model_name
        self.llm_model_name = llm_model_name
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.top_k = top_k

        self.chunks = []
        self.index = None
        self.full_text = ""

        print("LOADING EMBEDDING MODEL...")
        self.embedder = SentenceTransformer(embedding_model_name)

        print("LOADING TOKENIZER...")
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

        print("LOADING LLM...")
        if use_4bit and torch.cuda.is_available():
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4"
            )

            self.llm = AutoModelForCausalLM.from_pretrained(
                llm_model_name,
                device_map="auto",
                torch_dtype=torch.float16,
                quantization_config=quant_config
            )
        else:
            self.llm = AutoModelForCausalLM.from_pretrained(
                llm_model_name,
                device_map="auto",
                torch_dtype=torch.float32
            )

        self.llm.eval()
        print("STUDYMATE RAG READY")

    def load_pdf(self, pdf_path):
        doc = fitz.open(pdf_path)
        text_parts = []

        for page_num, page in enumerate(doc, start=1):
            text = page.get_text()
            if text.strip():
                text_parts.append(f"\n[PAGE {page_num}]\n{text}")

        doc.close()
        self.full_text = "\n".join(text_parts)
        return self.full_text

    def chunk_text(self, text):
        words = text.split()
        chunks = []
        start = 0
        chunk_id = 0

        while start < len(words):
            end = start + self.chunk_size
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)

            chunks.append({
                "chunk_id": chunk_id,
                "text": chunk_text
            })

            chunk_id += 1
            start += self.chunk_size - self.chunk_overlap

        return chunks

    def build_index(self, pdf_path):
        print("READING PDF...")
        text = self.load_pdf(pdf_path)

        print("CHUNKING TEXT...")
        self.chunks = self.chunk_text(text)

        print("TOTAL CHUNKS:", len(self.chunks))

        texts = [chunk["text"] for chunk in self.chunks]

        print("CREATING EMBEDDINGS...")
        embeddings = self.embedder.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        embeddings = np.array(embeddings).astype("float32")

        print("BUILDING FAISS INDEX...")
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        print("INDEX READY")

    def retrieve(self, query):
        query_embedding = self.embedder.encode(
            [query],
            normalize_embeddings=True,
            show_progress_bar=False
        )

        query_embedding = np.array(query_embedding).astype("float32")

        scores, indices = self.index.search(query_embedding, self.top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):
            chunk = dict(self.chunks[idx])
            chunk["score"] = float(score)
            results.append(chunk)

        return results

    def format_context(self, retrieved_chunks):
        context = ""

        for i, chunk in enumerate(retrieved_chunks, start=1):
            context += f"\n[CONTEXT {i}]\n"
            context += chunk["text"]
            context += "\n"

        return context

    def generate(self, system_prompt, user_prompt, max_new_tokens=500):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.llm.device)

        with torch.no_grad():
            output_ids = self.llm.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.3,
                do_sample=True
            )

        generated_ids = output_ids[0][len(inputs.input_ids[0]):]

        response = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        return response.strip()

    def summarize(self):
        context = " ".join([chunk["text"] for chunk in self.chunks[:8]])

        system_prompt = """
You are StudyMate AI, an academic study assistant.
Summarize uploaded study material clearly.
"""

        user_prompt = f"""
Summarize this uploaded file.

Include:
1. What the file is about
2. Main topics
3. Important concepts
4. What the student should review first

Text:
{context}
"""

        return self.generate(system_prompt, user_prompt, max_new_tokens=700)

    def answer_question(self, question):
        retrieved_chunks = self.retrieve(question)
        context = self.format_context(retrieved_chunks)

        system_prompt = """
You are StudyMate AI.
Answer only using the uploaded document context.
If the answer is not available, say you could not find it in the file.
"""

        user_prompt = f"""
Question:
{question}

Context:
{context}

Answer clearly:
"""

        return self.generate(system_prompt, user_prompt, max_new_tokens=500)

In [70]:
# ==========================================
# CELL 8 — INITIALIZE STUDYMATE RAG
# ==========================================

studymate = StudyMateRAG()

LOADING EMBEDDING MODEL...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

LOADING TOKENIZER...
LOADING LLM...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

STUDYMATE RAG READY


In [71]:
# ==========================================
# CELL 9 — BUILD INDEX FROM UPLOADED PDF
# ==========================================

studymate.build_index(pdf_path)

READING PDF...
CHUNKING TEXT...
TOTAL CHUNKS: 2
CREATING EMBEDDINGS...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

BUILDING FAISS INDEX...
INDEX READY


In [ ]:
# ==========================================
# CELL 10 — TEST PDF SUMMARY
# ==========================================

summary = studymate.summarize()

print("PDF SUMMARY:\n")
print(summary)

In [72]:
# ==========================================
# CELL 11 — TEST QUESTION ANSWERING
# ==========================================

question = "What is this document about?"

answer = studymate.answer_question(question)

print("QUESTION:")
print(question)

print("\nANSWER:\n")
print(answer)

QUESTION:
What is this document about?

ANSWER:

This document is an introduction to Natural Language Processing (NLP), covering its history, symbolic and statistical approaches, modern methods like neural networks, applications areas, and current trends such as transformer architectures and large language models. It also discusses various components of NLP including lexical analysis, syntactic parsing, semantic analysis, and machine translation. The content provides a comprehensive overview of NLP's development, methodologies, and practical applications across diverse domains.


In [74]:
# ==========================================
# CELL 12 — INTERACTIVE QUESTION ANSWERING
# ==========================================

question = input("ASK A QUESTION FROM THE FILE: ")

answer = studymate.answer_question(question)

print("\nANSWER:\n")
print(answer)

ASK A QUESTION FROM THE FILE: EXPLAIN THE BREIF HISTORY OF NLP

ANSWER:

The brief history of NLP can be summarized as follows:

1940s-1950s: Markov processes and finite-state machines were used for grammar.

1957-1970: Symbolic and stochastic approaches were developed.

1970-1983: Stochastic paradigms, logical paradigms, and discourse modeling emerged.

1983-1993: Finite-state models became prevalent.

2000-2007: The rise of machine learning was significant.

2017-present: Transformers revolutionized NLP with models like BERT, GPT, and others.


In [85]:
# ==========================================
# CELL 13 — STRICT QUIZ FROM PDF ONLY
# ==========================================

import re

def keep_exact_number_of_questions(text, num_questions):
    pattern = r'(\*\*Question\s+\d+:\*\*.*?)(?=\n---|\n\*\*Question\s+\d+:\*\*|$)'
    matches = re.findall(pattern, text, re.DOTALL)

    if len(matches) >= num_questions:
        return "\n---\n".join(matches[:num_questions])

    return text


def generate_quiz(difficulty="mixed", num_questions=5):

    retrieved_chunks = studymate.retrieve("main important concepts definitions examples")
    context = studymate.format_context(retrieved_chunks)

    system_prompt = """
You are StudyMate AI.

Generate quiz questions ONLY from the uploaded PDF.
Follow the exact number requested.
Do not create extra questions.
"""

    user_prompt = f"""
Create EXACTLY {num_questions} quiz questions from the uploaded PDF.

Difficulty: {difficulty}

Use this format only:

**Question 1:**
Question text

A) Option
B) Option
C) Option
D) Option

Answer: A

Rules:
- EXACTLY {num_questions} questions
- Stop after Question {num_questions}
- Do not create Question {num_questions + 1}
- Use only uploaded PDF content
- No title
- No conclusion
- No extra notes

Context:
{context}
"""

    raw_quiz = studymate.generate(
        system_prompt,
        user_prompt,
        max_new_tokens=1200
    )

    final_quiz = keep_exact_number_of_questions(raw_quiz, num_questions)

    return final_quiz

In [86]:
# ==========================================
# CELL 14 — TEST EASY QUIZ
# ==========================================

quiz = generate_quiz(
    difficulty="easy",
    num_questions=5
)

print(quiz)

**Question 1:**
What are some common methods used in machine learning classifiers for sentence boundary detection?

A) Bag-of-Words model  
B) Support Vector Machines  
C) Decision Trees  
D) Random Forest  

Answer: B) Support Vector Machines

---
**Question 2:**
Which aspect of sentence boundary detection is most challenging when dealing with mixed-language texts?

A) Determining the correct part of speech  
B) Identifying standard punctuation marks  
C) Accounting for non-Roman alphabets  
D) Handling multi-word expressions  

Answer: C) Accounting for non-Roman alphabets  

---
**Question 3:**
In which scenario would subword tokenization (e.g., BPE, WordPiece) be particularly beneficial?

A) When dealing with logographic writing systems  
B) When analyzing single-word sentences  
C) When performing semantic analysis  
D) When generating response-like outputs  

Answer: A) When dealing with logographic writing systems  

---
**Question 4:**
How does modern preprocessing, such as nor

In [87]:
# ==========================================
# CELL 15 — TEST MEDIUM QUIZ
# ==========================================

quiz = generate_quiz(
    difficulty="medium",
    num_questions=5
)

print(quiz)

**Question 1:**
What constitutes a word?

A) A group of letters that can stand alone as a meaningful unit
B) Any string of characters that follows standard punctuation rules
C) A single morpheme that cannot be broken down further
D) An arbitrary set of symbols used in programming

Answer: A

---
**Question 2:**
In sentence segmentation, what is typically done first?

A) Breaking sentences into smaller segments
B) Identifying individual words within sentences
C) Determining the part of speech for each word
D) Applying statistical methods to predict sentence boundaries

Answer: B

---
**Question 3:**
Which of the following best describes the modern approach to handling non-space delimited words in sentence segmentation?

A) Using regular expressions to match word boundaries
B) Employing machine learning classifiers to detect sentence boundaries
C) Implementing traditional rule-based methods
D) Utilizing lexical analysis to identify word boundaries

Answer: B

---
**Question 4:**
How many

In [88]:
# ==========================================
# CELL 16 — TEST HARD QUIZ
# ==========================================

quiz = generate_quiz(
    difficulty="hard",
    num_questions=5
)

print(quiz)

**Question 1:**
In what way does modern preprocessing differ significantly from traditional methods?

A) It involves normalization, lemmatization, and stemming.
  
   **Answer:** A

---
**Question 2:**
How does sentence boundary detection address the issue of standard punctuation?

A) By identifying the presence of period, question mark, and exclamation points.

   **Answer:** A

---
**Question 3:**
Which method is used to identify words in modern approaches?

A) Morphological analysis.

   **Answer:** A

---
**Question 4:**
What is an example of a modern application of NLP?

A) Sentiment analysis of social media posts.

   **Answer:** A

---
**Question 5:**
Which component of statistical NLP focuses on converting text into manipulable objects?

A) Corpus creation.

   **Answer:** A


In [89]:
# ==========================================
# CELL 17 — INTERACTIVE QUIZ MODE
# ==========================================

difficulty = input("ENTER DIFFICULTY easy / medium / hard / mixed: ")
num_questions = int(input("ENTER NUMBER OF QUESTIONS: "))

quiz = generate_quiz(
    difficulty=difficulty,
    num_questions=num_questions
)

print("\nGENERATED QUIZ:\n")
print(quiz)

ENTER DIFFICULTY easy / medium / hard / mixed: HARD
ENTER NUMBER OF QUESTIONS: 7

GENERATED QUIZ:

**Question 1:**
In which part of the NLP pipeline does lexical analysis occur?

A) Tokenization
B) Lexical analysis
C) Syntactic analysis
D) Semantic analysis

Answer: B

---
**Question 2:**
Which method involves breaking text into sentences based on standard punctuation like period, question mark, and exclamation point?

A) Sentence boundary detection
B) Character encoding issues
C) Contextual factors
D) Syntactic parsing

Answer: A

---
**Question 3:**
How many distinct methods are there for breaking text into sentences according to modern practices?

A) 2
B) 3
C) 4
D) 5

Answer: C

---
**Question 4:**
In what way does machine learning classifiers help in sentence boundary detection?

A) By identifying the end of sentences through statistical analysis
B) By detecting the start of sentences through pattern matching
C) By determining the importance of sentences through semantic analysis
D

In [90]:
# ==========================================
# CELL 18 — EXAM QUESTION GENERATOR
# ==========================================

import re

def keep_exact_exam_questions(text, num_questions):
    pattern = r'(\*\*Question\s+\d+:\*\*.*?)(?=\*\*Question\s+\d+:\*\*|$)'
    matches = re.findall(pattern, text, re.DOTALL)

    if len(matches) >= num_questions:
        return "\n".join(matches[:num_questions])

    return text


def generate_exam_questions(difficulty="mixed", num_questions=5):

    retrieved_chunks = studymate.retrieve(
        "important concepts definitions formulas repeated ideas likely exam questions"
    )

    context = studymate.format_context(retrieved_chunks)

    system_prompt = """
You are StudyMate AI.

Generate likely exam questions from the uploaded file.

Focus on:
- Important concepts
- Definitions
- Common exam topics
- Conceptual understanding
- Application based questions

Do not create extra questions.
"""

    user_prompt = f"""
Create EXACTLY {num_questions} possible exam questions.

Difficulty: {difficulty}

Format:

**Question 1:**
Question text

Answer:
Short model answer

Rules:
- EXACTLY {num_questions} questions
- Stop after Question {num_questions}
- No extra notes
- Use only uploaded file context
- Questions should feel like university exam questions

Context:
{context}
"""

    raw_output = studymate.generate(
        system_prompt,
        user_prompt,
        max_new_tokens=1600
    )

    return keep_exact_exam_questions(raw_output, num_questions)

In [91]:
# ==========================================
# CELL 19 — TEST EXAM QUESTIONS
# ==========================================

exam_q = generate_exam_questions(
    difficulty="easy",
    num_questions=5
)

print(exam_q)

**Question 1:**
**Question text:** 
Define the term "word" according to linguistics.

**Answer:**
In linguistics, a word is a meaningful unit of language consisting of one or more morphemes. It can stand alone as a complete phrase or function as part of a larger phrase or clause. Words typically convey specific meanings related to concepts such as people, places, things, actions, states, qualities, etc., and they combine through morphological processes to form longer words or phrases. 

### 
**Question 2:**
**Question text:** 
Explain the difference between sentence boundary detection and context in sentence segmentation.

**Answer:**
Sentence boundary detection involves identifying the beginning and end of sentences within a piece of text. This is important for applications such as search engines, where it helps to determine which sections of text contain relevant information. On the other hand, context in sentence segmentation refers to the surrounding elements of a sentence that hel

In [92]:
# ==========================================
# CELL 20 — INTERACTIVE EXAM MODE
# ==========================================

difficulty = input("ENTER DIFFICULTY easy / medium / hard / mixed: ")
num_questions = int(input("ENTER NUMBER OF QUESTIONS: "))

exam_q = generate_exam_questions(
    difficulty=difficulty,
    num_questions=num_questions
)

print("\nPOSSIBLE EXAM QUESTIONS:\n")
print(exam_q)

ENTER DIFFICULTY easy / medium / hard / mixed: HARD
ENTER NUMBER OF QUESTIONS: 7

POSSIBLE EXAM QUESTIONS:

**Question 1:**
What are some common challenges faced during the preprocessing steps of natural language processing?

**Answer:**
Some common challenges faced during the preprocessing steps of natural language processing include handling non-Roman alphabets such as Russian and Arabic, dealing with mixed-language texts where different scripts coexist, managing case distinctions in uppercase/lowercase patterns, resolving ambiguities in multi-word expressions, distinguishing between different parts of speech, and accurately identifying abbreviations and titles. Additionally, modern applications like social media and informal text require specialized techniques to handle emojis, contractions, and other stylistic elements effectively.

---

### 
**Question 2:**
Describe the role of normalization in the preprocessing of natural language data.

**Answer:**
Normalization plays a crucial 

In [96]:
# ==========================================
# CELL 21 — STRICT FLASHCARD GENERATOR
# ==========================================

def generate_one_flashcard(card_num):

    retrieved_chunks = studymate.retrieve(
        "important definitions key terms concepts examples"
    )

    context = studymate.format_context(retrieved_chunks)

    system_prompt = """
You are StudyMate AI.

Create one revision flashcard only.
Use uploaded file only.
"""

    user_prompt = f"""
Create Flashcard {card_num}

Format exactly:

**Flashcard {card_num}:**
Front: question / key term
Back: short answer

Use only this context:

{context}
"""

    return studymate.generate(
        system_prompt,
        user_prompt,
        max_new_tokens=250
    )


def generate_flashcards(num_cards=10):

    output = ""

    for i in range(1, num_cards + 1):
        card = generate_one_flashcard(i)
        output += card + "\n\n"

    return output

In [97]:
# ==========================================
# CELL 22 — TEST FLASHCARDS
# ==========================================

flashcards = generate_flashcards(num_cards=10)

print(flashcards)

**Flashcard 1:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information.**

**Flashcard 2:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information such as morphological analysis which includes word parts and combination rules.**

**Flashcard 3:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information such as morphological analysis which includes word parts and combination rules.**

**Flashcard 4:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information such as morphological analysis which includes word parts and combination rules.**

**Flashcard 5:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information.**

**Flashcard 6:**
Front: **What constitutes a word?**
Back: **Wo

In [98]:
# ==========================================
# CELL 23 — INTERACTIVE FLASHCARD MODE
# ==========================================

num_cards = int(input("ENTER NUMBER OF FLASHCARDS: "))

flashcards = generate_flashcards(num_cards=num_cards)

print("\nFLASHCARDS:\n")
print(flashcards)

ENTER NUMBER OF FLASHCARDS: 7

FLASHCARDS:

**Flashcard 1:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information such as morphological analysis which includes word parts and combination rules.**

**Flashcard 2:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information such as morphological analysis which includes word parts and combination rules.**

**Flashcard 3:**
Front: **Definition of a Word**
Back: A **unit of meaning**, consisting of letters or symbols that can stand alone as part of a sentence or phrase.

**Flashcard 4:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful information such as morphological analysis which includes word parts and combination rules.**

**Flashcard 5:**
Front: **What constitutes a word?**
Back: **Words are not atomic units. Word decomposition uncovers useful informati

In [100]:
# ==========================================
# CELL 24 — INSTALL YOUTUBE LIBRARIES
# ==========================================

!pip install -q youtube-transcript-api yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.6 MB/s eta 0:00:00


In [101]:
# ==========================================
# CELL 25 — IMPORT YOUTUBE LIBRARIES
# ==========================================

from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse, parse_qs
import yt_dlp

In [102]:
# ==========================================
# CELL 26 — EXTRACT YOUTUBE VIDEO ID
# ==========================================

def extract_youtube_id(url):
    parsed_url = urlparse(url)

    if parsed_url.hostname in ["www.youtube.com", "youtube.com"]:
        return parse_qs(parsed_url.query).get("v", [None])[0]

    if parsed_url.hostname == "youtu.be":
        return parsed_url.path[1:]

    return None

In [107]:
# ==========================================
# CELL 27 — FIXED GET YOUTUBE TRANSCRIPT
# ==========================================

def get_youtube_transcript(url):

    video_id = extract_youtube_id(url)

    if video_id is None:
        return "INVALID YOUTUBE URL"

    try:
        transcript = YouTubeTranscriptApi().fetch(video_id)

        text = " ".join([item.text for item in transcript])

        return text

    except Exception as e:
        return f"TRANSCRIPT NOT AVAILABLE: {e}"

In [108]:
# ==========================================
# CELL 28 — BUILD YOUTUBE RAG INDEX
# ==========================================

def build_youtube_index(url):
    transcript_text = get_youtube_transcript(url)

    if transcript_text.startswith("TRANSCRIPT NOT AVAILABLE"):
        print(transcript_text)
        return

    studymate.full_text = transcript_text
    studymate.chunks = studymate.chunk_text(transcript_text)

    texts = [chunk["text"] for chunk in studymate.chunks]

    embeddings = studymate.embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    embeddings = np.array(embeddings).astype("float32")

    dim = embeddings.shape[1]
    studymate.index = faiss.IndexFlatIP(dim)
    studymate.index.add(embeddings)

    print("YOUTUBE TRANSCRIPT INDEX READY")
    print("TOTAL CHUNKS:", len(studymate.chunks))

In [112]:
# ==========================================
# CELL 29 — TEST YOUTUBE SUMMARY
# ==========================================

youtube_url = input("PASTE YOUTUBE LINK: ")

build_youtube_index(youtube_url)

summary = studymate.summarize()

print("\nYOUTUBE SUMMARY:\n")
print(summary)

PASTE YOUTUBE LINK: https://www.youtube.com/watch?v=ZnBF2GeAKbo


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

YOUTUBE TRANSCRIPT INDEX READY
TOTAL CHUNKS: 3

YOUTUBE SUMMARY:

**What the File Is About:**
The file discusses the concept of algorithms, their role in various applications such as social media, search engines, and dating apps, and the potential ethical implications of algorithmic decision-making.

**Main Topics:**
- **Algorithms**: Definition, types (e.g., deterministic vs. probabilistic), and examples.
- **Search Engines**: How algorithms rank web pages and handle queries.
- **Social Media**: Algorithmic filtering and personalization.
- **Dating Apps**: Algorithm-driven matching and recommendations.
- **Ethical Considerations**: Concerns regarding transparency, bias, and job displacement due to automation.

**Important Concepts:**
- **Algorithm**: A step-by-step procedure to solve a problem or achieve a specific goal.
- **Deterministic vs. Probabilistic Algorithms**: Different approaches to handling uncertainty.
- **Ranking Algorithms**: Techniques used by search engines to order r

In [113]:
# ==========================================
# CELL 30 — ASK QUESTION FROM YOUTUBE VIDEO
# ==========================================

question = input("ASK A QUESTION FROM THE VIDEO: ")

answer = studymate.answer_question(question)

print("\nANSWER:\n")
print(answer)

ASK A QUESTION FROM THE VIDEO: WHAT IS THE VIDEO ABOUT ?

ANSWER:

The video discusses algorithms and their role in various applications such as drones, social media, and criminal justice systems. It explains that algorithms are sets of instructions processed by computers to perform tasks, similar to recipes or formulas. The video highlights the importance of transparency in algorithms, particularly concerning pricing for flights, and questions whether algorithms can think independently. It concludes by emphasizing the need for responsible usage of algorithms and the control of the data they operate on.


In [114]:
# ==========================================
# CELL 31 — GENERATE QUIZ FROM YOUTUBE VIDEO
# ==========================================

difficulty = input("ENTER DIFFICULTY easy / medium / hard / mixed: ")
num_questions = int(input("ENTER NUMBER OF QUESTIONS: "))

quiz = generate_quiz(
    difficulty=difficulty,
    num_questions=num_questions
)

print("\nVIDEO QUIZ:\n")
print(quiz)

ENTER DIFFICULTY easy / medium / hard / mixed: MIXED
ENTER NUMBER OF QUESTIONS: 7

VIDEO QUIZ:

**Question 1:**
What is an algorithm?

A) A process or set of rules to be followed in calculations or other problem-solving operations, especially by a computer  
B) A methodical approach to solve a problem  
C) A type of software program designed to automate tasks  
D) A sequence of steps performed by a computer  

Answer: A

---
**Question 2:**
How do algorithms work according to the given context?

A) By following a list of instructions  
B) By performing complex mathematical calculations  
C) By randomly selecting data points  
D) By comparing and contrasting information  

Answer: A

---
**Question 3:**
In the context provided, why are algorithms important?

A) To improve efficiency and accuracy in decision-making  
B) To increase the complexity of computations  
C) To ensure privacy and security  
D) To reduce the need for human intervention  

Answer: A

---
**Question 4:**
Which stat

In [118]:
# ==========================================
# CELL 32 — INSTALL VIDEO + WHISPER LIBRARIES
# ==========================================

!pip install -q openai-whisper moviepy

In [119]:
# ==========================================
# CELL 33 — IMPORT VIDEO LIBRARIES
# ==========================================

import whisper
import moviepy.editor as mp

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [120]:
# ==========================================
# CELL 34 — LOAD WHISPER MODEL
# ==========================================

whisper_model = whisper.load_model("base")

print("WHISPER MODEL LOADED")

WHISPER MODEL LOADED


In [121]:
# ==========================================
# CELL 35 — UPLOAD VIDEO FILE
# ==========================================

uploaded = files.upload()

video_folder = f"{base_path}/video"
os.makedirs(video_folder, exist_ok=True)

for file in uploaded.keys():
    shutil.move(file, f"{video_folder}/{file}")

video_path = f"{video_folder}/{list(uploaded.keys())[0]}"

print("VIDEO FILE UPLOADED:")
print(video_path)

Saving video_test.mp4 to video_test.mp4
VIDEO FILE UPLOADED:
/content/drive/MyDrive/STUDYMATE_AI/video/video_test.mp4


In [122]:
# ==========================================
# CELL 36 — EXTRACT AUDIO FROM VIDEO
# ==========================================

audio_from_video_path = f"{video_folder}/extracted_audio.wav"

video = mp.VideoFileClip(video_path)
video.audio.write_audiofile(audio_from_video_path)

print("AUDIO EXTRACTED FROM VIDEO:")
print(audio_from_video_path)

MoviePy - Writing audio in /content/drive/MyDrive/STUDYMATE_AI/video/extracted_audio.wav


MoviePy - Done.
AUDIO EXTRACTED FROM VIDEO:
/content/drive/MyDrive/STUDYMATE_AI/video/extracted_audio.wav


In [123]:
# ==========================================
# CELL 37 — TRANSCRIBE VIDEO AUDIO
# ==========================================

result = whisper_model.transcribe(audio_from_video_path)

video_text = result["text"]

print("VIDEO TRANSCRIPTION COMPLETE\n")
print(video_text[:3000])

VIDEO TRANSCRIPTION COMPLETE

 If I have to study DSC in 2026, this is how my planner would look like. In the very first month, I would cover areas, string, binary search and stack. And I will not randomly solve any question. I will solve with the patterns. So if we will learn questions through patterns now, then we will be able to identify whenever we see new questions. Okay, this question belongs to which pattern? Then the second month, I will start with linguists. Then I will do heap, then HashMap. Design HashMap type question, I'll mostly ask them interview. And then recursion. Because to start with DP, we should start with recursion. And in the third and fourth month, I will do tree with DFS and BFS traversants. Then graph also with DFS and BFS traversants. Because I have learned recursion in my second month, so I can do this, then I will go with DP. And then after that, I will go with try and bit manipulation at the end. And after this, I would like to go with greedy and path tra

In [124]:
# ==========================================
# CELL 38 — BUILD VIDEO RAG INDEX
# ==========================================

studymate.full_text = video_text
studymate.chunks = studymate.chunk_text(video_text)

texts = [chunk["text"] for chunk in studymate.chunks]

embeddings = studymate.embedder.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

dim = embeddings.shape[1]

studymate.index = faiss.IndexFlatIP(dim)
studymate.index.add(embeddings)

print("VIDEO INDEX READY")
print("TOTAL CHUNKS:", len(studymate.chunks))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

VIDEO INDEX READY
TOTAL CHUNKS: 1


In [125]:
# ==========================================
# CELL 39 — VIDEO SUMMARY
# ==========================================

summary = studymate.summarize()

print("VIDEO SUMMARY:\n")
print(summary)

VIDEO SUMMARY:

**What the File Is About:**
The file outlines a study plan for preparing for Data Structures and Algorithms (DSC) in 2026. The main focus is on systematically covering key topics such as string algorithms, binary search, stacks, recursion, design of data structures like HashMaps, and dynamic programming problems.

**Main Topics:**
- **String Algorithms**: Techniques for manipulating strings efficiently.
- **Binary Search**: Searching for elements within sorted arrays or lists.
- **Stacks**: Implementations and operations on stack data structures.
- **Recursion**: Solving problems using recursive functions.
- **Design Patterns**: Common problem-solving strategies and their applications.
- **Dynamic Programming (DP)**: Solving optimization problems by breaking them into subproblems.
- **Data Structures**: Implementation and use of various data structures like HashMaps.
- **Heaps**: Priority queues implemented using heaps.
- **Graph Traversal**: Depth First Search (DFS) an

In [126]:
# ==========================================
# CELL 40 — ASK QUESTION FROM VIDEO
# ==========================================

question = input("ASK A QUESTION FROM VIDEO: ")

answer = studymate.answer_question(question)

print("\nANSWER:\n")
print(answer)

ASK A QUESTION FROM VIDEO: what is the video about and how long is the video ?

ANSWER:

The video is about a study plan for Data Structures and Algorithms (DSC) in 2026. The speaker plans to cover various topics such as areas, strings, binary search, stacks, linguistics, heaps, hash maps, recursion, dynamic programming (DP), trees, depth-first search (DFS), breadth-first search (BFS), graphs, and more. They suggest solving problems based on identified patterns to improve understanding of new concepts. The video does not specify the length of the video but mentions covering multiple months of study.


In [127]:
# ==========================================
# CELL 41 — QUIZ FROM VIDEO
# ==========================================

difficulty = input("ENTER DIFFICULTY easy / medium / hard / mixed: ")
num_questions = int(input("ENTER NUMBER OF QUESTIONS: "))

quiz = generate_quiz(
    difficulty=difficulty,
    num_questions=num_questions
)

print("\nVIDEO QUIZ:\n")
print(quiz)

ENTER DIFFICULTY easy / medium / hard / mixed: EASY 
ENTER NUMBER OF QUESTIONS: 5

VIDEO QUIZ:

**Question 1:**
What is the most common pattern for solving problems related to recursion?

A) Stack
B) Binary Search
C) String
D) Heap

Answer: A

---
**Question 2:**
In which month of your planner would you start learning about design questions involving Hash Maps?

A) First Month
B) Second Month
C) Third Month
D) Fourth Month

Answer: B

---
**Question 3:**
Which topic would you focus on during the third month according to your planner?

A) Tree traversal (DFS/BFS)
B) Graph traversal (DFS/BFS)
C) Greedy Algorithms
D) Path Tracking

Answer: A

---
**Question 4:**
During the fourth month, what would you prioritize based on your planner's schedule?

A) Try and Bit Manipulation
B) Dynamic Programming (DP)
C) Linguistics
D) Heap

Answer: B

---
**Question 5:**
According to your planner, by the end of the fifth month, you would likely cover topics such as **greedy algorithms**, **path tracking*

In [128]:
# ==========================================
# CELL 42 — BOOK RECOMMENDATION FUNCTION
# ==========================================

def recommend_books(problem_statement, num_books=5):

    system_prompt = """
You are StudyMate AI, an academic book recommendation assistant.

Recommend books based on the user's learning problem.
Prioritize:
- Relevance to the problem
- Beginner friendliness
- Common academic use
- High reputation
- Practical usefulness

Do not invent fake ratings.
If exact ratings are not available, say "rating not verified".
"""

    user_prompt = f"""
User problem:
{problem_statement}

Recommend {num_books} books.

Format:

1. Book Title
Author:
Best for:
Why recommended:
Difficulty:
Rating note:

Keep it clean and useful.
"""

    return studymate.generate(
        system_prompt,
        user_prompt,
        max_new_tokens=900
    )

In [129]:
# ==========================================
# CELL 43 — TEST BOOK RECOMMENDATION
# ==========================================

problem = """
I am struggling with Natural Language Processing basics.
I want to understand tokenization, embeddings, transformers, and RAG.
"""

books = recommend_books(problem, num_books=5)

print("BOOK RECOMMENDATIONS:\n")
print(books)

BOOK RECOMMENDATIONS:

1. **Natural Language Processing: Theory, Technology, and Applications** by Christopher Manning, Prabhakar Ramabhadran, Hinrich Schütze
   - Author: Christopher Manning, Prabhakar Ramabhadran, Hinrich Schütze
   - Best for: Beginners looking to understand NLP concepts.
   - Why recommended: This book provides a comprehensive introduction to NLP theory, technology, and applications, making it ideal for beginners.
   - Difficulty: Easy
   - Rating note: The rating is not verified as this book is widely considered one of the best in its field.

2. **Deep Learning** by Ian Goodfellow, Yoshua Bengio, Aaron Courville
   - Author: Ian Goodfellow, Yoshua Bengio, Aaron Courville
   - Best for: Understanding the foundational concepts of deep learning that are crucial for understanding NLP.
   - Why recommended: This book covers the essential topics such as neural networks, backpropagation, and optimization algorithms, which are fundamental to understanding NLP models like 

In [131]:
# ==========================================
# CELL 44 — INTERACTIVE BOOK RECOMMENDER
# ==========================================

problem_statement = input("DESCRIBE YOUR LEARNING PROBLEM: ")
num_books = int(input("HOW MANY BOOKS DO YOU WANT?: "))

books = recommend_books(
    problem_statement,
    num_books=num_books
)

print("\nRECOMMENDED BOOKS:\n")
print(books)

DESCRIBE YOUR LEARNING PROBLEM: I WANT TO LEARN NLP. I WANT YOUR TOP 5 BEGINNER FRIENDLY BOOK RECCOMENDATION
HOW MANY BOOKS DO YOU WANT?: 5

RECOMMENDED BOOKS:

1. **Natural Language Processing: Theory and Implementation** by Christopher Manning, Prabhakar Ramabhadran, Hinrich Schütze (2014)
   - Author: Christopher Manning, Prabhakar Ramabhadran, Hinrich Schütze
   - Best for: Beginners in Natural Language Processing (NLP) who want a comprehensive understanding of theory and practical implementation.
   - Why recommended: This book provides a solid foundation in NLP concepts through clear explanations and numerous examples. It covers both theoretical aspects and practical implementations using Python libraries like NLTK and spaCy.
   - Difficulty: Intermediate
   - Rating note: Highly recommended for beginners looking to build a strong conceptual framework in NLP.

2. **Deep Learning** by Ian Goodfellow, Yoshua Bengio, Aaron Courville (2016)
   - Author: Ian Goodfellow, Yoshua Bengio,

In [136]:
# ==========================================
# CELL 49 — ROADMAP TREE GENERATOR FUNCTION
# ==========================================

def generate_roadmap_tree(topic, depth="medium", level="beginner"):

    system_prompt = """
You are StudyMate AI, a learning roadmap generator.

Create a clear tree-style roadmap for the user's topic.
Make it structured, practical, and easy to follow.
"""

    user_prompt = f"""
Create a learning roadmap as a tree.

Topic: {topic}
Depth: {depth}
Level: {level}

Format like this:

TOPIC
├── 1. Main Area
│   ├── Subtopic
│   ├── Subtopic
│   └── Practice Task
├── 2. Main Area
│   ├── Subtopic
│   └── Practice Task
└── Final Project Idea

Rules:
- Use tree format
- Beginner = simple foundations
- Intermediate = include tools and implementation
- Advanced = include research, optimization, and projects
- Depth can be short, medium, or deep
- Keep it useful for a student
"""

    return studymate.generate(
        system_prompt,
        user_prompt,
        max_new_tokens=1200
    )

In [138]:
# ==========================================
# CELL 50 — TEST ROADMAP TREE
# ==========================================

roadmap = generate_roadmap_tree(
    topic="Trees in Data Structures",
    depth="deep",
    level="beginner"
)

print("ROADMAP TREE:\n")
print(roadmap)

ROADMAP TREE:

**Learning Roadmap**

**Topic:** Trees in Data Structures  
**Depth:** Deep  
**Level:** Beginner  

---

### **1. Introduction to Trees**
- **Subtopic:** What is a Tree?
  - Definition of trees in data structures.
  - Visual representation of different types of trees (e.g., binary, AVL).
  - Basic properties of trees.
- **Practice Task:** Create a visual representation of a tree using an online tool or software.
- **Final Project Idea:** Design a simple game that uses a tree structure to represent levels or objectives.

---

### **2. Binary Trees**
- **Subtopic:** Properties of Binary Trees
  - In-order, pre-order, and post-order traversal.
  - Balanced vs. unbalanced trees.
  - Height and depth of a tree.
- **Practice Task:** Implement a function to perform an in-order traversal on a given binary tree.
- **Final Project Idea:** Develop a sorting algorithm that utilizes a balanced binary search tree.

---

### **3. Binary Search Trees**
- **Subtopic:** Operations in BST

In [139]:
# ==========================================
# CELL 51 — INTERACTIVE ROADMAP TREE
# ==========================================

topic = input("ENTER TOPIC: ")
depth = input("ENTER DEPTH short / medium / deep: ")
level = input("ENTER LEVEL beginner / intermediate / advanced: ")

roadmap = generate_roadmap_tree(
    topic=topic,
    depth=depth,
    level=level
)

print("\nROADMAP TREE:\n")
print(roadmap)

ENTER TOPIC: MACHINE LEARNING 
ENTER DEPTH short / medium / deep: SHORT
ENTER LEVEL beginner / intermediate / advanced: BEGINNER

ROADMAP TREE:

MACHINE LEARNING
├── 1. Introduction to Machine Learning
    ├── 1.1 What is Machine Learning?
        ├── 1.1.1 Definition
        ├── 1.1.2 Examples
    ├── 1.2 Types of Machine Learning
        ├── 1.2.1 Supervised Learning
        ├── 1.2.2 Unsupervised Learning
        ├── 1.2.3 Reinforcement Learning
    ├── 1.3 Applications of Machine Learning
        ├── 1.3.1 Healthcare
        ├── 1.3.2 Finance
        ├── 1.3.3 Retail
        ├── 1.3.4 Manufacturing
        ├── 1.3.5 Transportation
        ├── 1.3.6 Education
        ├── 1.3.7 Cybersecurity
        ├── 1.3.8 Environmental Science
        ├── 1.3.9 Social Media
        ├── 1.3.10 Entertainment
        ├── 1.3.11 Other Applications
    ├── 1.4 Importance of Machine Learning
        ├── 1.4.1 Efficiency
        ├── 1.4.2 Accuracy
        ├── 1.4.3 Scalability
        ├── 1.4.4 Innovati